#### Prompt base (multiuso — análise + código)

In [61]:
entrada_usuario = "Explique o que é um DataFrame em pandas"

In [62]:
prompt_ollama = f"""
Você é um assistente técnico especializado em Python, ciência de dados e análise de sistemas.

Tarefa:
Analise o conteúdo abaixo e responda de forma clara, objetiva e estruturada.

Se for código:
- Explique o que ele faz
- Identifique erros ou problemas
- Sugira melhorias mínimas

Se for texto:
- Resuma
- Extraia insights principais
- Sugira aplicações práticas

Formato da resposta:
1. Resumo
2. Diagnóstico
3. Sugestões

Conteúdo:
<<<
{entrada_usuario}
>>>
"""

#### Prompt focado em geração de código

In [63]:
prompt_codigo = f"""
Você é um especialista em Python e engenharia de dados.

Objetivo:
Gerar código funcional e direto ao ponto.

Regras:
- Não explique demais
- Gere código pronto para execução
- Use boas práticas
- Evite dependências desnecessárias

Entrada:
{entrada_usuario}

Saída esperada:
Código Python completo e funcional.
"""

#### Prompt focado em análise de texto

In [64]:
prompt_analise = f"""
Você é um pesquisador em ciência de dados e análise textual.

Tarefa:
Analise o texto abaixo com rigor técnico.

Inclua:
- Resumo técnico
- Principais conceitos
- Possíveis aplicações
- Limitações

Texto:
<<<
{entrada_usuario}
>>>
"""

#### Exemplo de chamada para o Ollama (API)

#### Diagnóstico - verifica qual modelo responde

In [85]:
import requests

urls = [
    "http://ollama:11434",
    "http://ollama:11434/api/tags",
    "http://ollama:11434/api/generate",
    "http://simserv-ollama-ui:11434",
    "http://simserv-ollama-ui:11434/api/tags",
]

for url in urls:
    try:
        r = requests.get(url, timeout=5)
        print("\nURL:", url)
        print("STATUS:", r.status_code)
        print("RESPOSTA:", r.text[:300])
    except Exception as e:
        print("\nURL:", url)
        print("ERRO:", e)


URL: http://ollama:11434
STATUS: 200
RESPOSTA: Ollama is running

URL: http://ollama:11434/api/tags
STATUS: 200
RESPOSTA: {"models":[{"name":"deepseek-coder:33b","model":"deepseek-coder:33b","modified_at":"2026-03-18T16:01:28.250035226Z","size":18819456424,"digest":"acec7c0b0fd9ad4c9e82620b32dcb794120d40623ab53d5e1453a948d7d204d8","details":{"parent_model":"","format":"gguf","family":"llama","families":["llama"],"param

URL: http://ollama:11434/api/generate
STATUS: 405
RESPOSTA: 405 method not allowed

URL: http://simserv-ollama-ui:11434
ERRO: HTTPConnectionPool(host='simserv-ollama-ui', port=11434): Max retries exceeded with url: / (Caused by NewConnectionError("HTTPConnection(host='simserv-ollama-ui', port=11434): Failed to establish a new connection: [Errno 111] Connection refused"))

URL: http://simserv-ollama-ui:11434/api/tags
ERRO: HTTPConnectionPool(host='simserv-ollama-ui', port=11434): Max retries exceeded with url: /api/tags (Caused by NewConnectionError("HTTPConnection(host

#### Verifica quais modelos disponíveis

In [93]:
def listar_modelos_ollama():
    import socket
    socket.setdefaulttimeout(900) # Força o socket a ter paciência
    
    for base in ["http://ollama:11434"]:
        try:
            # Adicionamos headers explicitamente para manter a conexão viva
            req = Request(base + "/api/tags", headers={"Connection": "keep-alive"})
            with urlopen(req, timeout=900) as resposta:
                dados = json.loads(resposta.read().decode("utf-8"))
            return base, [m.get("name") for m in dados.get("models", [])]
        except Exception as e:
            print(f"Tentativa falhou: {e}")
    return None, []

In [94]:
listar_modelos_ollama()

('http://ollama:11434',
 ['deepseek-coder:33b',
  'qwen2.5-coder:32b',
  'qwen2.5-coder:14b',
  'deepseek-coder:6.7b'])

In [95]:
import json
from urllib.request import urlopen, Request


urls = [
    "http://ollama:11434",
    "http://127.0.0.1:11434",
    "http://localhost:11434",
]

def carregar_modelo_na_memoria(modelo):
    for base in urls:
        try:
            payload = {
                "model": modelo,
                "prompt": " ",
                "stream": False
            }

            req = Request(
                f"{base}/api/generate",
                data=json.dumps(payload).encode("utf-8"),
                headers={"Content-Type": "application/json"}
            )

            urlopen(req, timeout=600)

            print(f"Modelo {modelo} carregado via {base}")
            return True

        except Exception as erro:
            print(f"Falhou em {base}: {erro}")

    print(f"Não foi possível pré-carregar o modelo {modelo}")
    return False
# uso
modelo = carregar_modelo_na_memoria('qwen2.5-coder:32b')
print(modelo)

Modelo qwen2.5-coder:32b carregado via http://ollama:11434
True


In [84]:
import json
import socket
from urllib.request import urlopen, Request

def carregar_modelo_na_memoria(modelo):
    payload = {"model": modelo}
    req = Request(
        "http://ollama:11434/api/generate", # Endpoint de geração
        data=json.dumps(payload).encode("utf-8"),
        headers={"Content-Type": "application/json"}
    )
    try:
        # Enviamos o pedido e não esperamos a resposta completa (apenas o sinal de recebido)
        urlopen(req, timeout=5) 
    except Exception:
        print(f"Modelo {modelo} está sendo carregado em segundo plano...")

# Use antes de consultar
model = carregar_modelo_na_memoria("qwen2.5-coder:32b")
model 

In [99]:
# Função principal (com preload integrado)
def chamar_ollama(prompt, model="deepseek-coder:33b"):
    
    # 1. Pré-carrega modelo (não bloqueante crítico)
    carregar_modelo_na_memoria(model)

    # 2. Tenta gerar resposta
    for base in urls:
        try:
            response = requests.post(
                f"{base}/api/generate",
                json={
                    "model": model,
                    "prompt": prompt,
                    "stream": False
                },
                timeout=60  # ↑ importante: modelos grandes demoram
            )

            if response.status_code == 200:
                return response.json().get("response", "")

        except Exception:
            continue

    return "Erro: não foi possível conectar ao Ollama"

# uso
resposta = chamar_ollama(prompt, model="deepseek-coder:33b")
print(resposta)

Modelo deepseek-coder:33b carregado via http://ollama:11434


Erro: não foi possível conectar ao Ollama


In [100]:
prompt = "Explique o que é um DataFrame em pandas"

respostas = chamar_ollama(prompt, model="qwen2.5-coder:32b")

print(respostas)

Modelo qwen2.5-coder:32b carregado via http://ollama:11434


Erro: não foi possível conectar ao Ollama


#### Exemplos de configuração Markdown

#### Prompt Engine + Fallback de infraestrutura

- tenta várias URLs (resiliência)
- adapta tipo de tarefa (texto vs código)
- estrutura resposta (padronização)

#### Prompt Engine + Fallback de infraestrutura
tenta várias URLs (resiliência)<br>
adapta tipo de tarefa (texto vs código)<br>
estrutura resposta (padronização)

#### Exemplo de Funções em python

In [102]:
PROMPTS = {
    "codigo": prompt_codigo,
    "analise": prompt_analise,
    "geral": prompt_ollama
}

def gerar_prompts(tipo, entrada):
    prompt = PROMPTS.get(tipo, PROMPTS["geral"])
    return prompt.format(entrada_usuario=entrada)

In [101]:
def gerar_prompt(tipo, entrada):
    if tipo == "codigo":
        return prompt_codigo.format(entrada_usuario=entrada)
    elif tipo == "analise":
        return prompt_analise.format(entrada_usuario=entrada)
    else:
        return prompt_ollama.format(entrada_usuario=entrada)

  

In [107]:
resultado = gerar_prompts(
    entrada="Explique DataFrame",
    tipo="analise"
)
resultado

'\nVocê é um pesquisador em ciência de dados e análise textual.\n\nTarefa:\nAnalise o texto abaixo com rigor técnico.\n\nInclua:\n- Resumo técnico\n- Principais conceitos\n- Possíveis aplicações\n- Limitações\n\nTexto:\n<<<\nExplique o que é um DataFrame em pandas\n>>>\n'

In [112]:
gerar_prompt(prompt_codigo, entrada_usuario)

'\nVocê é um assistente técnico especializado em Python, ciência de dados e análise de sistemas.\n\nTarefa:\nAnalise o conteúdo abaixo e responda de forma clara, objetiva e estruturada.\n\nSe for código:\n- Explique o que ele faz\n- Identifique erros ou problemas\n- Sugira melhorias mínimas\n\nSe for texto:\n- Resuma\n- Extraia insights principais\n- Sugira aplicações práticas\n\nFormato da resposta:\n1. Resumo\n2. Diagnóstico\n3. Sugestões\n\nConteúdo:\n<<<\nExplique o que é um DataFrame em pandas\n>>>\n'

In [114]:
#entrada_usuario = "Explique o que é um DataFrame em pandas"
prompt = prompt_analise.format(entrada_usuario=entrada_usuario)
print(prompt)


Você é um pesquisador em ciência de dados e análise textual.

Tarefa:
Analise o texto abaixo com rigor técnico.

Inclua:
- Resumo técnico
- Principais conceitos
- Possíveis aplicações
- Limitações

Texto:
<<<
Explique o que é um DataFrame em pandas
>>>



#### Gerar código

In [115]:
entrada = "crie uma função Python que soma dois números"

prompt = gerar_prompt("codigo", entrada)

print(prompt)


Você é um especialista em Python e engenharia de dados.

Objetivo:
Gerar código funcional e direto ao ponto.

Regras:
- Não explique demais
- Gere código pronto para execução
- Use boas práticas
- Evite dependências desnecessárias

Entrada:
Explique o que é um DataFrame em pandas

Saída esperada:
Código Python completo e funcional.



In [ ]:
entrada = "crie uma função Python que soma dois números"

prompt = gerar_prompt("codigo", entrada)

resposta = chamar_ollama(prompt, model="qwen2.5-coder:32b")

print(resposta)

In [117]:
entrada = "crie uma função Python que soma dois números"

prompt = gerar_prompt("codigo", entrada)

resposta = chamar_ollama(prompt, model="qwen2.5-coder:32b")

print(resposta)

Modelo qwen2.5-coder:32b carregado via http://ollama:11434


```python
import pandas as pd

# Criando um exemplo de DataFrame
data = {
    'Nome': ['Alice', 'Bob', 'Charlie'],
    'Idade': [25, 30, 35],
    'Cidade': ['São Paulo', 'Rio de Janeiro', 'Belo Horizonte']
}

df = pd.DataFrame(data)

# Exibindo o DataFrame
print(df)
```


#### Analisar texto

In [116]:
entrada = "O uso de IA na educação está crescendo rapidamente."

prompt = gerar_prompt("analise", entrada)

print(prompt)


Você é um pesquisador em ciência de dados e análise textual.

Tarefa:
Analise o texto abaixo com rigor técnico.

Inclua:
- Resumo técnico
- Principais conceitos
- Possíveis aplicações
- Limitações

Texto:
<<<
Explique o que é um DataFrame em pandas
>>>



In [31]:
entrada = "crie um código para ler CSV com pandas"

prompt = gerar_prompt("codigo", entrada)

resposta = chamar_ollama(prompt)

print(resposta)